In [1]:
!pip -q install pymupdf sentence-transformers faiss-cpu openai numpy

In [2]:
import os
import fitz
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from openai import OpenAI

from google.colab import files, userdata

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [3]:
uploaded = files.upload()

pdf_path = next(iter(uploaded.keys()))

print("Uploaded file:", pdf_path)

Saving Campus_Library_Guide.pdf to Campus_Library_Guide.pdf
Uploaded file: Campus_Library_Guide.pdf


In [4]:
doc = fitz.open(pdf_path)

documents = []

for page_number, page in enumerate(doc, start=1):

    text = page.get_text("text").strip()

    if text:
        documents.append({
            "page": page_number,
            "text": text
        })

print("Total pages extracted:", len(documents))

for item in documents:
    print(f"\n--- PAGE {item['page']} ---")
    print(item["text"])

Total pages extracted: 2

--- PAGE 1 ---
Campus Library Guide
A sample document for practising Retrieval-Augmented Generation (RAG)
Library Hours
The campus library is open Monday to Friday from 8:00 AM to 8:00 PM. On Saturdays, it is open from 9:00
AM to 5:00 PM. The library is closed on Sundays and public holidays.
Membership
All currently enrolled students and faculty members may use the library. Students must show their campus ID
card when borrowing books. Visitors may use reading spaces but cannot borrow books.
Borrowing Rules
Students may borrow up to four books at a time. The standard loan period is 14 days. Faculty may borrow up
to ten books for 30 days.
Renewals
A student may renew a book once for an additional seven days if no other reader has reserved it. Renewal
requests must be made before the original due date.
Late Returns
A late fee of 2 rupees per book per day applies to student loans. Books overdue by more than 30 days may
result in borrowing privileges being temporar

In [5]:
total_characters = sum(
    len(item["text"]) for item in documents
)

print("Total pages:", len(documents))
print("Total characters:", total_characters)

if total_characters > 0:
    print("PDF extraction successful!")
else:
    print("No text found in PDF.")

Total pages: 2
Total characters: 2074
PDF extraction successful!


In [6]:
chunk_size = 500
overlap = 100

chunks = []

for item in documents:

    words = item["text"].split()

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk_text = " ".join(words[start:end])

        if chunk_text.strip():
            chunks.append({
                "text": chunk_text,
                "page": item["page"]
            })

        start += chunk_size - overlap

print("Total chunks created:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1} | Page {chunk['page']}")
    print(chunk["text"][:200])

Total chunks created: 2

Chunk 1 | Page 1
Campus Library Guide A sample document for practising Retrieval-Augmented Generation (RAG) Library Hours The campus library is open Monday to Friday from 8:00 AM to 8:00 PM. On Saturdays, it is open f

Chunk 2 | Page 2
The first floor is designated as a quiet study zone. Group discussion is permitted only in the group study rooms and the designated collaborative area. Contact For library support, email library.suppo


In [7]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [8]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)
print("Embeddings generated successfully!")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (2, 384)
Embeddings generated successfully!


In [9]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created successfully!")
print("Total vectors stored:", index.ntotal)

FAISS index created successfully!
Total vectors stored: 2


In [10]:
def retrieve_documents(question, top_k=4):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        min(top_k, len(chunks))
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        results.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "score": float(score)
        })

    return results

In [11]:
question = "What are the library opening hours?"

results = retrieve_documents(question)

for result in results:

    print("Page:", result["page"])
    print("Similarity:", round(result["score"], 3))
    print("Content:", result["text"])
    print("-" * 50)

Page: 1
Similarity: 0.553
Content: Campus Library Guide A sample document for practising Retrieval-Augmented Generation (RAG) Library Hours The campus library is open Monday to Friday from 8:00 AM to 8:00 PM. On Saturdays, it is open from 9:00 AM to 5:00 PM. The library is closed on Sundays and public holidays. Membership All currently enrolled students and faculty members may use the library. Students must show their campus ID card when borrowing books. Visitors may use reading spaces but cannot borrow books. Borrowing Rules Students may borrow up to four books at a time. The standard loan period is 14 days. Faculty may borrow up to ten books for 30 days. Renewals A student may renew a book once for an additional seven days if no other reader has reserved it. Renewal requests must be made before the original due date. Late Returns A late fee of 2 rupees per book per day applies to student loans. Books overdue by more than 30 days may result in borrowing privileges being temporarily su

In [12]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("Groq API connected successfully!")

Groq API connected successfully!


In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Hello, introduce yourself."}
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

Hello! I’m ChatGPT, a conversational AI developed by OpenAI. I’m here to help answer questions, brainstorm ideas, explain concepts, or just chat about almost anything you’re curious about. Whether you need help with writing, coding, learning a new topic, or just want a friendly conversation, I’ve got you covered. How can I assist you today?


In [13]:
def ask_rag(question):

    results = retrieve_documents(question, top_k=4)

    if not results:
        return "No relevant information found in the PDF."

    context = ""

    for i, result in enumerate(results):
        context += f"""
        Context {i+1}
        Page Number: {result['page']}
        Content: {result['text']}
        """

    system_prompt = """
    You are a helpful PDF question-answering assistant.

    Answer ONLY using the provided PDF context.

    Rules:
    1. Do not use outside knowledge.
    2. Do not invent facts.
    3. If information is unavailable, say:
       Information not available in the uploaded document.
    4. Provide clear answers.
    5. Mention source page numbers.
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": f"""
                PDF CONTEXT:
                {context}

                QUESTION:
                {question}
                """
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [14]:
question = "How many books can a student borrow?"

answer = ask_rag(question)

print("Question:", question)
print("\nAnswer:", answer)

Question: How many books can a student borrow?

Answer: A student may borrow up to **four books at a time**【Context 1, Page 1】.


In [15]:
test_questions = [
    "How many books can a student borrow?",
    "What are the library opening hours?",
    "What is the late return fee?",
    "How many study rooms are available?",
    "What is the library support email?"
]

for question in test_questions:
    print("\nQuestion:", question)
    print("Answer:", ask_rag(question))
    print("-" * 60)


Question: How many books can a student borrow?
Answer: A student may borrow up to **four books at a time**【Context 1, Page 1】.
------------------------------------------------------------

Question: What are the library opening hours?
Answer: The campus library is open:

- **Monday to Friday:** 8:00 AM – 8:00 PM  
- **Saturday:** 9:00 AM – 5:00 PM  
- **Closed on Sundays and public holidays**【Context 1, Page 1】
------------------------------------------------------------

Question: What is the late return fee?
Answer: The late return fee is **2 rupees per book per day** for student loans【Context 1, Page 1】.
------------------------------------------------------------

Question: How many study rooms are available?
Answer: The library has **six** group study rooms. (Source: Context 2, Page 1)
------------------------------------------------------------

Question: What is the library support email?
Answer: The library support email is **library.support@example.edu**【Context 2, Page 2】
--

In [16]:
question = "Does the library provide free laptops?"

print(ask_rag(question))

Information not available in the uploaded document.
